# Course Q&A RAG System

This notebook walks through a course-question answering pipeline: document extraction, chunking, multilingual embeddings, persistent vector search, grounded generation, source citations, and evaluation.

The application includes three course folders: Deep Learning, Data Analysis, and Parallel and Distributed Computing. It reads PDF, PPTX, DOCX, CSV, and TXT files.

Pipeline: Course files → Extract and clean → Split into chunks → Embed → Persistent Chroma index → Retrieve → Answer with citations

## Project layout

Keep this notebook and `course_rag.py` in the project folder. Course files are grouped by course under `data/courses`.

```text
data/courses/
  Deep_Learning/
  Data_Analysis/
  Parallel_Distributed_Computing/
```

Add each new lecture file to the matching course folder. The folder name becomes the course label used in search filters and citations.

In [ ]:
%pip install -r requirements.txt

## 1. Configure the pipeline

The vector database is stored on disk in `data/chroma_db`. The embedding model supports multilingual search, and the language model is loaded when the first answer is requested.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from course_rag import CourseRAG, RAGConfig

config = RAGConfig(
    course_dir=PROJECT_DIR / "data" / "courses",
    chroma_dir=PROJECT_DIR / "data" / "chroma_db",
    chunk_size=800,
    chunk_overlap=120,
    top_k=4,
    required_courses=3,
)

rag = CourseRAG(config)

## 2. Build the searchable course index

Run this cell after adding or changing course files. It reports the number of files and text chunks indexed per course. Files that cannot be read are reported as warnings.

In [ ]:
report = rag.ingest()
print(f"Files indexed: {report['files']}")
print(f"Text chunks indexed: {report['chunks']}")
print("Chunks by course:")
for course, count in sorted(report["courses"].items()):
    print(f"  {course}: {count}")
if report["warnings"]:
    print("\nFiles needing attention:")
    for warning in report["warnings"]:
        print(f"  {warning}")

## 3. Course coverage and retrieval evaluation

The first chart shows how many chunks were indexed for each course. To evaluate retrieval, add real student questions and their expected course names to `evaluation_questions.csv`. The notebook then measures course-level hit@k and charts the observed results.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

coverage = pd.DataFrame(
    [{"Course": course, "Chunks": count} for course, count in sorted(report["courses"].items())]
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(coverage["Course"], coverage["Chunks"], color="#286A8E")
ax.set_title("Indexed text chunks by course")
ax.set_ylabel("Number of chunks")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
plt.show()

evaluation_path = PROJECT_DIR / "evaluation_questions.csv"
evaluation_cases = pd.read_csv(evaluation_path).dropna(subset=["question", "expected_course"])
if not evaluation_cases.empty:
    evaluation_rows = []
    for row in evaluation_cases.itertuples(index=False):
        retrieved = rag.retrieve(str(row.question), top_k=config.top_k)
        hit = any(item["metadata"]["course"] == str(row.expected_course) for item in retrieved)
        evaluation_rows.append({
            "Case": f"Q{len(evaluation_rows) + 1}",
            "Question": str(row.question),
            "Expected course": str(row.expected_course),
            "Hit@k": int(hit),
        })
    evaluation_results = pd.DataFrame(evaluation_rows)
    display(evaluation_results)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(evaluation_results["Case"], evaluation_results["Hit@k"] * 100, color="#3A8D6D")
    ax.set_title(f"Expected-course retrieval hit@{config.top_k}")
    ax.set_ylabel("Hit (%)")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    plt.show()
else:
    print("Add test questions and expected course names to evaluation_questions.csv to generate the retrieval evaluation chart.")

## 4. Ask a course question

The answer is generated from the retrieved passages. Each cited source includes its course, file, and page, slide, row, or chunk location. If the retrieved material does not support an answer, the system abstains.

In [ ]:
question = input("Enter a question about the course materials: ").strip()
result = rag.answer(question, top_k=config.top_k)
print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(
        f"{source['citation']} {source['course']} | {source['relative_path']} | "
        f"{source['location']} | similarity={source['similarity']:.3f}"
    )

## 5. Run the Streamlit application

From the project folder, install `requirements.txt` and run:

```powershell
python -m streamlit run app.py
```

Use **Build or refresh index** in the sidebar after changing course materials. The Streamlit app includes course filters, source citations, and charts for course coverage and retrieved-source similarity.